In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

dt = pd.read_csv('../data/processed/train_data.csv')

In [2]:
object_columns = dt.select_dtypes(include=['object']).columns

for col in object_columns:
    dt[col] = dt[col].str.lower().str.strip()

print("Log: Text standardization (lowercase & strip) completed for all object columns.")

Log: Text standardization (lowercase & strip) completed for all object columns.


In [ ]:
hidden_missing_values = ['unknown', 'none', 'not defined', 'null', 'nan', '']
dt[object_columns] = dt[object_columns].replace(hidden_missing_values, np.nan)

print(f"Log: Hidden missing values {hidden_missing_values} converted to actual np.nan.")

Log: Hidden missing values ['unknown', 'none', 'not defined', 'null', 'nan', ''] converted to actual np.nan.


In [4]:
dt.describe(include='object').T

,count,unique,top,freq
ProductCD,590540,5,w,439670
card4,588963,4,visa,384767
card6,588969,4,debit,439938
P_emaildomain,496084,59,gmail.com,228355
R_emaildomain,137291,60,gmail.com,57147
M4,309096,3,m0,196405
id_23,5169,3,ip_proxy:transparent,3489
id_30,77565,75,windows 10,21155
id_31,140282,130,chrome 63.0,22000
id_33,73289,260,1920x1080,16874


In [5]:
#Identify numeric columns
numeric_cols = dt.select_dtypes(include=['int64', 'float64']).columns

#Find potential categorical columns based on unique value count
potential_categorical = []
threshold = 100 

for col in numeric_cols:
    unique_count = dt[col].nunique()
    
    #Exclude the target variable from this list to prevent data leakage
    if unique_count < threshold and col != 'isFraud':
        potential_categorical.append(col)

print(f"Found {len(potential_categorical)} disguised categorical columns.")

#Convert them to 'category' data type
dt[potential_categorical] = dt[potential_categorical].astype('category')

print("Type casting completed. Memory usage optimized.")

Found 267 disguised categorical columns.
Type casting completed. Memory usage optimized.


In [6]:
v_138_166 = [f'V{i}' for i in range(138, 167)]
v_322_339 = [f'V{i}' for i in range(322, 340)]
id_series = [f'id_{str(i).zfill(2)}' for i in [24, 25, 7, 8, 21, 26, 27, 23, 22, 18, 4, 3, 33, 9, 10, 30, 32, 34, 14]]

other_cols = ['dist2', 'D7', 'D13', 'D14', 'D12', 'D6', 'D8', 'D9','addr2', 'DeviceInfo', 'TransactionID', 'TransactionDT']

high_null_cols = v_138_166 + v_322_339 + id_series + other_cols

dt['TransactionAmt_Log'] = np.log1p(dt['TransactionAmt'])

dt1 = dt.drop(columns=list(high_null_cols) + ['TransactionAmt'])

In [7]:
dt1.head(5)

,isFraud,ProductCD,card1,card2,card3,card4,card5,card6,addr1,P_emaildomain,...,id_12,id_16,id_28,id_29,id_35,id_36,id_37,id_38,DeviceType,TransactionAmt_Log
0,0,w,13926,NaN,150.0,discover,142.0,credit,315.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.241327
1,0,w,2755,404.0,150.0,mastercard,102.0,credit,325.0,gmail.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.401197
2,0,w,4663,490.0,150.0,visa,166.0,debit,330.0,outlook.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.094345
3,0,w,18132,567.0,150.0,mastercard,117.0,debit,476.0,yahoo.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.931826
4,0,h,4497,514.0,150.0,mastercard,102.0,credit,420.0,gmail.com,...,144.0,0.0,0.0,samsung browser 6.2,1.0,0.0,1.0,1.0,1.0,3.931826


In [8]:
# --- MISSING VALUE ANALYSIS ---
print("--- MISSING VALUE ANALYSIS ---")

# Calculate missing value percentages for all columns
missing_values = dt1.isnull().sum()
missing_percentages = (missing_values / len(dt1)) * 100

# Create a DataFrame for better viewing
missing_df = pd.DataFrame({'Missing_Count': missing_values, 'Missing_Percentage': missing_percentages})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Percentage', ascending=False)

print(f"Total columns with missing values: {len(missing_df)} out of {dt.shape[1]}")
print("\nTop 15 columns with most missing values:")
display(missing_df.head(15))

--- MISSING VALUE ANALYSIS ---
Total columns with missing values: 338 out of 435

Top 15 columns with most missing values:


,Missing_Count,Missing_Percentage
V49,168969,28.612626
id_15,461200,78.098012
id_16,461200,78.098012
V52,168969,28.612626
V278,460110,77.913435
V277,460110,77.913435
V39,168969,28.612626
V40,168969,28.612626
V41,168969,28.612626
V42,168969,28.612626


In [9]:
dt1.head()

,isFraud,ProductCD,card1,card2,card3,card4,card5,card6,addr1,dist1,...,id_20,id_28,id_29,id_31,id_35,id_36,id_37,id_38,DeviceType,TransactionAmt_Log
0,0,w,13926,NaN,150.0,discover,142.0,credit,315.0,19.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.241327
1,0,w,2755,404.0,150.0,mastercard,102.0,credit,325.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.401197
2,0,w,4663,490.0,150.0,visa,166.0,debit,330.0,287.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.094345
3,0,w,18132,567.0,150.0,mastercard,117.0,debit,476.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.931826
4,0,h,4497,514.0,150.0,mastercard,102.0,credit,420.0,NaN,...,144.0,0.0,0.0,samsung browser 6.2,1.0,0.0,1.0,1.0,1.0,3.931826


In [ ]:
#dt1.to_csv('../data/processed/train_cleaned.csv', index=False)